### Biblioteki

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter


In [2]:
from src.models.load_data import split_data, create_dataloader, create_dataset, get_sampler
from src.utils.load_data import load_data
from src.music_dataset.MusicDataset import MusicDataset

### Zmienn globalne

In [3]:
TRAIN_PATH = "data/train.pkl"
TEST_PATH = "data/test_no_target.pkl"
BOUNDRY = 1024
PAD_VALUE = 0

### Device

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Podział danych

In [5]:
train_data = load_data(TRAIN_PATH)

In [9]:
x_train, y_train, x_val, y_val, x_test, y_test = split_data(train_data, train_ratio=0.75)

train_dataset = create_dataset(x_train, y_train, BOUNDRY, PAD_VALUE)
val_dataset = create_dataset(x_val, y_val, BOUNDRY, PAD_VALUE)
test_dataset = create_dataset(x_test, y_test, BOUNDRY, PAD_VALUE)

In [14]:
train_sampler = get_sampler(train_dataset)
val_sampler = get_sampler(val_dataset)
test_sampler = get_sampler(test_dataset)

train_dataloader = create_dataloader(train_dataset, batch_size=32, sampler=train_sampler)
val_dataloader = create_dataloader(val_dataset, batch_size=32, sampler=val_sampler)
test_dataloader = create_dataloader(test_dataset, batch_size=32, sampler=test_sampler)

In [15]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"\nTrain dataloader size: {len(train_dataloader)}")
print(f"Validation dataloader size: {len(val_dataloader)}")
print(f"Test dataloader size: {len(test_dataloader)}")

Train dataset size: 1653
Validation dataset size: 551
Test dataset size: 735

Train dataloader size: 52
Validation dataloader size: 18
Test dataloader size: 23


### Wagi klas

In [16]:
classes = [c[1].item() for c in train_dataset.samples]
class_counts = Counter(classes)
num_classes = len(class_counts)
total_count = sum(class_counts.values())

# Compute inverse frequency weights and normalize by number of classes
class_weights = {cls: total_count / (num_classes * count) for cls, count in class_counts.items()}
class_weights_tensor = torch.tensor(list(class_weights.values()), dtype=torch.float32, device=device)

print(f"Class weights: {class_weights_tensor}")

Class weights: tensor([0.3609, 1.3331, 1.2290, 3.8000, 2.4857], device='cuda:0')


### Model

In [33]:
import torch
import torch.nn as nn

class GRUClassifier(nn.Module):
    def __init__(
        self,
        num_acords,
        embedding_dim=32,
        hidden_size=128,
        output_size=5,
        num_layers=2,
        dropout=0.3,
        bidirectional=True
    ):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=num_acords, embedding_dim=embedding_dim, padding_idx=0)

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=bidirectional
        )

        factor = 2 if bidirectional else 1
        self.norm = nn.LayerNorm(hidden_size * factor)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * factor, output_size)

    def forward(self, x):
        x = x.long()
        x = self.embedding(x)  # -> (batch_size, seq_len, embedding_dim)
        out, _ = self.gru(x)   # -> (batch_size, seq_len, hidden_size * factor)

        # Mean pooling across sequence length
        pooled = out.mean(dim=1)  # -> (batch_size, hidden_size * factor)

        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)

        logits = self.fc(pooled)  # -> (batch_size, output_size)
        return logits


In [35]:
from tqdm import tqdm
NUM_ACCORDS = 194
model = GRUClassifier(num_acords=NUM_ACCORDS).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(120):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_dataloader, desc="Training"):
        inputs, targets = batch
        inputs, targets = inputs.to(device), targets.to(device)


        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    train_loss /= len(train_dataloader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc="Validation"):
            inputs, targets = batch
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
    val_loss /= len(val_dataloader)

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [53]:
model.eval()
class_correct = {}
class_total = {}

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing"):
        inputs, targets = batch
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)

        for label in torch.unique(targets):
            mask = targets == label
            count = mask.sum().item()
            correct = (predictions[mask] == targets[mask]).sum().item()
            # accumulate counts
            class_total[label.item()] = class_total.get(label.item(), 0) + count
            class_correct[label.item()] = class_correct.get(label.item(), 0) + correct

acc_per_class = {cls: class_correct[cls] / class_total[cls] for cls in class_total}
mean_acc = sum(acc_per_class.values()) / len(acc_per_class)

print("Accuracy per class:")
for cls, acc in acc_per_class.items():
    print(f"Class {cls}: {acc:.4f}")
print(f"\nMean Accuracy: {mean_acc:.4f}")

Testing: 100%|██████████| 23/23 [00:00<00:00, 47.13it/s]

Accuracy per class:
Class 0: 0.8131
Class 1: 0.6343
Class 2: 0.1389
Class 3: 0.7094
Class 4: 0.6538

Mean Accuracy: 0.5899
